# 12. PITWALL Replay Engine & Strategy Replay Pipeline

This notebook runs the end-to-end replay engine for session 9558 (Silverstone 2024, 52 laps).
It demonstrates:
1. Loading validated session data and detecting clean laps, pit in/out laps, and finishers.
2. Fitting the lap fixed-effects pace model to isolate compound and degradation effects without weather confounding.
3. Measuring empirical pit loss across historical green-flag stops.
4. Running the acceptance test suite (Tests A, B, C, D).
5. Re-running the original motivating scenarios: Driver 63 (Russell) vs Driver 44 (Hamilton) at lap 26.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pitwall.data import load_race, get_finishers, get_driver_laps
from pitwall.pace_model import PaceModel
from pitwall.pit_loss import measure_pit_loss
from pitwall.engine import replay, replay_actual, Stay, Pit
from pitwall.validation import run_plumbing_test, run_identity_test, run_sanity_bound_test, run_support_test

# 1. Load Session 9558 Data
df = load_race(9558)
finishers = get_finishers(df)
print(f'Loaded session 9558: {len(df)} lap records across {df["driver_number"].nunique()} drivers.')
print(f'Finishers ({len(finishers)}): {finishers}')
print(f'Clean laps identified: {df["is_clean_lap"].sum()}')

## 2. Fit Lap Fixed-Effects Pace Model

The model formula is:
$$\text{lap\_duration} \sim C(\text{lap\_number}) + C(\text{driver\_number}) + C(\text{compound}) + C(\text{compound}):\text{tyre\_age}$$
The per-lap term absorbs track wetness and fuel burn, preventing compound/tyre-age confounding.

In [ ]:
model = PaceModel(reference_compound='MEDIUM').fit(df)
print(f'Model R-squared: {model.rsquared:.4f}')
rep = model.identification_report
print(f'Clean laps: {rep.n_clean_laps}, Multi-compound laps: {rep.n_multi_compound_laps}')
print('\nCompound Pair Same-Lap Identification:')
for pair, cnt in rep.pair_counts.items():
    status = '[WEAK]' if pair in rep.weakly_identified_pairs else '[OK]'
    print(f'  {status} {pair}: {cnt} laps')

# Coefficient table
coef_data = []
for c in model.coefficients:
    if 'compound' in c.param_name.lower():
        coef_data.append({'Parameter': c.param_name, 'Estimate': round(c.coef, 4), '95% CI Lower': round(c.ci_lower, 4), '95% CI Upper': round(c.ci_upper, 4), 'p-value': round(c.pvalue, 4)})
pd.DataFrame(coef_data)

## 3. Empirical Pit-Loss Measurement

Pit loss is measured directly by comparing actual in-lap + out-lap times against expected clean pace from the model.

In [ ]:
green_summary, sc_summary = measure_pit_loss(df, model)
print(f'Green-flag pit stops measured: n={green_summary.n_stops}')
print(f'Median pit loss: {green_summary.median:.2f} s (IQR: [{green_summary.iqr_low:.2f}, {green_summary.iqr_high:.2f}] s)')
print(f'Mean pit loss: {green_summary.mean:.2f} s +/- {green_summary.std:.2f} s')

# Plot pit loss distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(green_summary.stops_table['pit_loss'], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(green_summary.median, color='red', linestyle='--', linewidth=2, label=f'Median ({green_summary.median:.2f}s)')
ax.axvline(22.0, color='orange', linestyle=':', linewidth=2, label='Old Assumption (22.0s)')
ax.set_title('Green-Flag Pit Loss Distribution (Session 9558)')
ax.set_xlabel('Pit Loss (s)')
ax.set_ylabel('Count')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Acceptance Test Suite

We execute Tests A, B, C, and D.

In [ ]:
res_a = run_plumbing_test(df)
res_b = run_identity_test(df, pace_model=model, pit_loss=green_summary.median)
res_c = run_sanity_bound_test(df, target_driver=44, decision_lap=20, pace_model=model, pit_loss=green_summary.median)
supp_ok, supp_msg = run_support_test(df, 'INTERMEDIATE', (28, 38))

test_results_summary = [
    {'Test': res_a.name, 'Status': 'PASS' if res_a.passed else 'FAIL', 'Metric': res_a.message},
    {'Test': res_b.name, 'Status': 'PASS' if res_b.passed else 'FAIL', 'Metric': res_b.message},
    {'Test': res_c.name, 'Status': 'PASS' if res_c.passed else 'FAIL', 'Metric': res_c.message},
    {'Test': 'Support Test (Test D)', 'Status': 'PASS' if supp_ok else 'FAIL', 'Metric': supp_msg}
]
pd.DataFrame(test_results_summary)

## 5. Decision Replay: Driver 63 (Russell) vs Driver 44 (Hamilton) at Lap 26

Original question: At lap 26, compare Stay Out vs Pitting for INTERMEDIATE on laps 27-31.

In [ ]:
# Scenario analysis for Driver 63 (Ground truth ends at lap 34)
r63_results = []
for p_lap in [27, 28, 29, 30, 31]:
    r = replay(df, 63, 26, Pit(lap=p_lap, compound='INTERMEDIATE'), model, green_summary.median)
    l34 = r.laps_table[(r.laps_table['driver_number']==63) & (r.laps_table['lap_number']==34)].iloc[0]
    r63_results.append({'Action': f'Pit Lap {p_lap} (INTER)', 'GT Available': r.ground_truth_available, 'Pos @ Lap 34': int(l34['position']), 'Cum Time @ Lap 34': round(float(l34['cum_time']), 2)})
r_stay = replay(df, 63, 26, Stay(), model, green_summary.median)
l34_stay = r_stay.laps_table[(r_stay.laps_table['driver_number']==63) & (r_stay.laps_table['lap_number']==34)].iloc[0]
r63_results.append({'Action': 'Stay Out', 'GT Available': r_stay.ground_truth_available, 'Pos @ Lap 34': int(l34_stay['position']), 'Cum Time @ Lap 34': round(float(l34_stay['cum_time']), 2)})
print('Driver 63 (Russell) Replay Evaluation (Retired Lap 34):')
pd.DataFrame(r63_results)

In [ ]:
# Scenario analysis for Driver 44 (Hamilton, Finisher)
r44_results = []
for p_lap in [27, 28, 29, 30, 31]:
    r = replay(df, 44, 26, Pit(lap=p_lap, compound='INTERMEDIATE'), model, green_summary.median)
    r44_results.append({'Action': f'Pit Lap {p_lap} (INTER)', 'Final Pos': r.summary['final_position'], 'Total Delta vs Act (s)': round(r.summary['total_time_delta_vs_actual'], 2), 'Pos Gain': r.summary['positions_gained_vs_actual']})
r44_stay = replay(df, 44, 26, Stay(), model, green_summary.median)
r44_results.append({'Action': 'Stay Out', 'Final Pos': r44_stay.summary['final_position'], 'Total Delta vs Act (s)': round(r44_stay.summary['total_time_delta_vs_actual'], 2), 'Pos Gain': r44_stay.summary['positions_gained_vs_actual']})
print('Driver 44 (Hamilton) Replay Evaluation (Full 52 Laps):')
pd.DataFrame(r44_results)

## 6. Gap-to-Leader vs Lap Comparison

Visualizing Hamilton's actual race vs counterfactual pit stop on Lap 27 vs Stay Out.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
res_pit27 = replay(df, 44, 26, Pit(lap=27, compound='INTERMEDIATE'), model, green_summary.median)
res_stay = replay(df, 44, 26, Stay(), model, green_summary.median)

ham_act = df[df['driver_number']==44]
ham_pit27 = res_pit27.laps_table[res_pit27.laps_table['driver_number']==44]
ham_stay = res_stay.laps_table[res_stay.laps_table['driver_number']==44]

leader_times = df.groupby('lap_number')['cum_time'].min().to_dict()
act_gaps = [ham_act[ham_act['lap_number']==l]['cum_time'].iloc[0] - leader_times[l] for l in range(1, 53)]

ax.plot(range(1, 53), act_gaps, label='Actual Race (Hamilton)', color='purple', linewidth=2)
ax.plot(ham_pit27['lap_number'], ham_pit27['gap_to_leader'], label='Counterfactual Pit Lap 27 (INTER)', color='teal', linestyle='--', linewidth=2)
ax.plot(ham_stay['lap_number'], ham_stay['gap_to_leader'], label='Counterfactual Stay Out', color='red', linestyle=':', linewidth=2)
ax.set_title('Gap to Leader vs Lap: Hamilton Actual vs Counterfactuals (Decision Lap 26)')
ax.set_xlabel('Lap Number')
ax.set_ylabel('Gap to Leader (s)')
ax.axvline(26, color='gray', linestyle='--', alpha=0.7, label='Decision Lap 26')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()